<a href="https://colab.research.google.com/github/fqixiang/workshop_llm_data_collection/blob/main/notebooks/llm_data_collection_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using Large Language Models for Data Collection/Annotation in Social Sciences and Humanities


## Package setup (SANE)

In [56]:
import os  # Gives access to operating system functionalities like file paths, environment variables, and directory operations.
from datetime import datetime  # Generates timestamps for log files.
import pandas as pd  # Imports the pandas library (aliased as pd) for handling and analyzing tabular data in DataFrames.
import numpy as np  # Imports NumPy (aliased as np), a library for fast numerical computations and array manipulations.
from tqdm import tqdm  # Imports tqdm, a progress bar utility that provides visual feedback for loops and long-running processes.
from typing import List # Imports the List type from the typing module for type annotations.
from langchain.chat_models import init_chat_model  # Imports a function from LangChain to initialize a chat-based large language model (LLM) interface.
from langchain_core.prompts import ChatPromptTemplate  # Imports a template class for creating structured prompts used in LLM interactions.
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field  # Imports BaseModel (for defining structured data models) and Field (for specifying metadata and validation rules).
import krippendorff  # Imports the krippendorff library, used to compute Krippendorff’s alpha — a reliability measure for agreement among raters or annotators.

## Data loading

We encourage you to use the datasets available in SANE. The default code reuses the toy dataset from the previous notebook. 

In [25]:
# Load CSV into a dataframe
data_url = "../data/srl_data_example.csv"
df = pd.read_csv(data_url)

Note that only the first 10 rows contain the anonymized text of the conversations. We will use these texts for the prompting experiments in this notebook.

In [26]:
# Use these 10 rows to define test_ids and test_conversations
test_ids = df.id[:10].tolist()
test_conversations = df.conversation[:10].tolist()

## Local open-weights deployment in SANE with Ollama

We will use Ollama to run open-weights models locally. 
Ollama is installed by default in the SANE environment, but you will need to pull a model and start the local server to use it.

Example commands (run in a command line terminal):

- `ollama list` (to see available models)
- `ollama pull <model_name>` (e.g., `ollama pull qwen2.5:7b` to pull the 7B version of the Qwen2.5 model; this only needs to be done once per model)
- `ollama serve`  (starts the local server at http://localhost:11434; skip if already running)

Currently, we have the following models available locally via Ollama:
- qwen2.5:7b
- qwen2.5:14b
- qwen2.5-coder:7b
- qwen2.5-coder:14b
- gpt-oss:20b

The number after the colon indicates the number of parameters in the model (e.g., `qwen2.5:7b` has 7 billion parameters). Larger numbers indicate more parameters, hence bigger and more powerful (but often slower) models. The "coder" models are optimized for code generation, which may be useful for coding support and debugging.

In [33]:
# Set up your model
model_name = "qwen2.5:7b"
temperature = 0  
max_tokens = 1000
seed = 123

# To prompt a self-hosted Ollama model, we simply point LangChain to the local server.
model = ChatOllama(
    model=model_name,
    base_url="http://localhost:11434",
    temperature=temperature,
    num_predict=max_tokens,
    seed=seed,
)

## Working with a single prompt

Let's start with the system prompt (i.e., high-level instruction to the model).

In [35]:
# Define a system prompt that explains the task and scoring rubric
system_prompt = """
You are an expert in educational assessment and goal evaluation, with
specialized expertise in applying deductive coding schemes to score the quality
and content of student goals.

##TASK##
A university student was given a series of prompts, guiding them through the
process of setting and elaborating on an academic goal for the coming week. You
will be provided with the entire conversation including the prompts, and the
student answers. Your objective is to assess the specificity of of the student’s
goal on a scale of 0 to 2 based on the entire conversation.
"""

Use the prompt template module `ChatPromptTemplate` from langchain to create a prompt request with both **system** and **user** prompts.

In [36]:
# Build a reusable prompt template with system + user parts
prompt_template = ChatPromptTemplate([
    ("system", system_prompt),
    ("user", "{conversation}"),
])

# Fill the template with the first conversation as a test case
single_prompt_request = prompt_template.invoke({"conversation": test_conversations[0]})

Check the prompt before prompting the model:

In [38]:
# View the prepared message list (system + user)
single_prompt_request.to_messages()

[SystemMessage(content='\nYou are an expert in educational assessment and goal evaluation, with\nspecialized expertise in applying deductive coding schemes to score the quality\nand content of student goals.\n\n##TASK##\nA university student was given a series of prompts, guiding them through the\nprocess of setting and elaborating on an academic goal for the coming week. You\nwill be provided with the entire conversation including the prompts, and the\nstudent answers. Your objective is to assess the specificity of of the student’s\ngoal on a scale of 0 to 2 based on the entire conversation.\n'),
 HumanMessage(content='CHATBOT: Set an academic goal for the upcoming week.\nSTUDENT: I would like to catch up on my geography reading\nCHATBOT: Add details to make your goal more specific.\nSTUDENT: I need to either read the book from last week and this week, or read my friends notes on the reading to take notes of my own so I dont fall behind.\nCHATBOT: How will you measure progress on and 

Prompt the model and inspect the response!

In [39]:
# Make the API call to get a single response
single_response = model.invoke(single_prompt_request)
print(single_response.content)

To assess the specificity of the student's goal, we will use a deductive coding scheme that evaluates the goal on three levels: clarity, measurability, and time-bound nature.

### Goal: "I would like to catch up on my geography reading."

1. **Clarity**: The goal is somewhat clear but lacks detail.
   - *Score*: 0 (out of 2)

2. **Measurability**: The student added a measure by stating they will track the number of pages read per day, which provides some level of measurability.
   - *Score*: 1 (out of 2)

3. **Time-bound Nature**: The goal is set for the upcoming week, providing a time frame.
   - *Score*: 1 (out of 2)

### Total Specificity Score: 2 out of 6

**Explanation:**
- **Clarity**: The initial goal was vague and lacked specific details about what exactly needs to be read or how much. Adding steps in the subsequent responses improved clarity but did not fully meet the criteria.
- **Measurability**: While the student added a measure (number of pages per day), it is still somewh

Voila! You have your first successful prompting interaction with an LLM API!

## Working with multiple prompts
Next, we go beyond a single prompt. Instead, we will work with **multiple prompts** at the same time.

**Tip**: Start with a small number of rows first to estimate time and (if using a paid API) cost to avoid surprises.

In [40]:
# Iterate over the conversations and score them
multiple_responses = {}
for id, conversation in tqdm(zip(test_ids, test_conversations),
                             total=len(test_ids),
                             desc="Processing Requests"):
    prompt_request = prompt_template.invoke({"conversation": conversation})
    # Call the model and store the full response for each conversation by ID
    response = model.invoke(prompt_request)
    multiple_responses[id] = response.content

Processing Requests:  10%|█         | 1/10 [00:48<07:15, 48.39s/it]


KeyboardInterrupt: 

Inspect the responses!

In [ ]:
# Show an example response
print(multiple_responses['chat_2'])

## Using structured output with a single prompt

Use the `BaseModel` and `Field` classes from the `pydantic` package to specify the desired output format, and the model will return an output that matches it.

For example:

In [ ]:
# Define the expected structured output schema
class SpecificityFormat(BaseModel):
    goal_specificity: int = Field(
        description="Score for goal specificity. Only return an integer from 0 to 2",
    )
    reasoning: str = Field(description="The reasoning to justify the score")

In [ ]:
# Build a prompt for a single conversation
prompt_request = prompt_template.invoke({"conversation": test_conversations[0]})

structured_model = model.with_structured_output(SpecificityFormat)
single_structured_response = structured_model.invoke(prompt_request)

In [ ]:
# Convert the structured response to a plain dict for inspection
dict(single_structured_response)

{'goal_specificity': 2,
 'reasoning': "The student's goal is specific as they have outlined clear actions (taking notes on particular chapters and articles) with measurable progress indicators (well-written notes summarizing important parts). The plan also includes a step-by-step approach to achieving the goal."}

## Using structured output with multiple prompts

Being able to work with multiple prompts at the same time and obtain structured output will save you a substantial amount of time in research projects!

In [46]:
# Score multiple conversations with structured output
multiple_structured_responses = {}
for id, conversation in tqdm(zip(test_ids, test_conversations),
                                     total=len(test_ids),
                                     desc="Processing Messages"):
    prompt_request = prompt_template.invoke({"conversation": conversation})
    structured_model = model.with_structured_output(SpecificityFormat)
    structured_response = structured_model.invoke(prompt_request)
    multiple_structured_responses[id] = structured_response

Processing Messages: 100%|██████████| 10/10 [01:04<00:00,  6.43s/it]


Display all the structured responses:

In [47]:
# Extract the scores from the structured responses
structured_scores = {id: resp.goal_specificity for id, resp in multiple_structured_responses.items()}
structured_scores

{'chat_1': 1,
 'chat_2': 2,
 'chat_3': 0,
 'chat_4': 2,
 'chat_5': 1,
 'chat_6': 1,
 'chat_7': 2,
 'chat_8': 1,
 'chat_9': 2,
 'chat_10': 2}

In [48]:
# Extract the reasoning text from the structured responses
structured_reasonings = {id: resp.reasoning for id, resp in multiple_structured_responses.items()}
structured_reasonings

{'chat_1': "The student's goal is somewhat specific as it outlines the need to catch up on reading and taking notes, but lacks clear details such as the exact number of pages or topics to cover. The plan includes steps but does not specify how many days or what resources will be used.",
 'chat_2': "The student's goal is specific as they have outlined clear actions (taking notes on particular chapters and articles) with measurable progress indicators (well-written notes summarizing important parts). The plan also includes a step-by-step approach to achieving the goal.",
 'chat_3': "The student's goal is not specific as it does not provide clear and measurable actions or outcomes.",
 'chat_4': "The student's goal is specific as it includes clear actions (finishing class, going to the gym, reading a quarter of the next day's readings), and measurable criteria (reading at least one quarter). The plan also provides steps for achieving the goal.",
 'chat_5': "The student's goal is somewhat s

## Enhancing reproducibility: log prompts and decisions

To make your data collection/annotation reproducible, log each prompt, model configuration, and model output. The block below writes CSV and JSONL logs to the `logs/` folder with a timestamped filename.

As these logs include prompts and model outputs, make sure to remove or anonymize sensitive information before sharing logs.

We show below how to log the prompts and responses from the structured output experiment, but you can apply the same logic to log any other prompting experiment you do in this notebook or in your own projects.

In [50]:
# Build a prompt + decision log
os.makedirs("logs", exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_csv_path = f"logs/prompt_log_{timestamp}.csv"
log_jsonl_path = f"logs/prompt_log_{timestamp}.jsonl"

# Score multiple conversations with structured output
multiple_structured_responses = {} # Store the structured responses
prompt_logs = [] # Store the prompt logs
for id, conversation in tqdm(zip(test_ids, test_conversations),
                                     total=len(test_ids),
                                     desc="Processing Messages"):

    # Ollama: structured output built in
    prompt_request = prompt_template.invoke({"conversation": conversation})
    structured_model = model.with_structured_output(SpecificityFormat)
    structured_response = structured_model.invoke(prompt_request)
    multiple_structured_responses[id] = structured_response

    # Log the prompt, model config, and output
    messages = prompt_request.to_messages()
    prompt_text = "\n\n".join([f"{m.type.upper()}: {m.content}" for m in messages])
    prompt_logs.append({
        "request_id": id,
        "provider": "ollama",
        "model": model_name,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "seed": seed,
        "prompt_text": prompt_text,
        "score": structured_scores.get(id),
        "reasoning": structured_reasonings.get(id),
    })

# Save logs to CSV and JSONL
prompt_log_df = pd.DataFrame(prompt_logs)
prompt_log_df.to_csv(log_csv_path, index=False)
prompt_log_df.to_json(log_jsonl_path, orient="records", lines=True)

# View the prompt log dataframe
prompt_log_df.head(3)

Processing Messages: 100%|██████████| 10/10 [01:04<00:00,  6.41s/it]


,request_id,provider,model,temperature,max_tokens,seed,prompt_text,score,reasoning
0,chat_1,ollama,qwen2.5:7b,0,1000,123,SYSTEM: \nYou are an expert in educational ass...,1,The student's goal is somewhat specific as it ...
1,chat_2,ollama,qwen2.5:7b,0,1000,123,SYSTEM: \nYou are an expert in educational ass...,2,The student's goal is specific as they have ou...
2,chat_3,ollama,qwen2.5:7b,0,1000,123,SYSTEM: \nYou are an expert in educational ass...,0,The student's goal is not specific as it does ...


## Check annotation quality

Implement a handy function to calculate Krippendorff's Alpha (i.e., agreement) between two lists of specificity scores.

In [ ]:
def compute_krippendorff_alpha(x: List[int], y: List[int]):
  # Format data into a reliability matrix (rows=raters, cols=items)
  data_krippendorff = np.array([x, y])
  # Compute Krippendorff’s Alpha (interval metric)
  kripp_alpha = krippendorff.alpha(reliability_data=data_krippendorff,
                                   level_of_measurement='ordinal')
  return kripp_alpha

Let's check the agreement between the specificity scores we got from the LLM above and the human expert-coded specificity scores!

In [53]:
# Compare agreement between expert and LLM ratings
expert_specificity_scores = df.expert_specificity_score[:10].tolist()
structured_llm_specificity_scores = list(structured_scores.values())
print("Krippendorff's Alpha:", compute_krippendorff_alpha(structured_llm_specificity_scores, expert_specificity_scores))

Krippendorff's Alpha: 0.5546606334841628


Not a great agreement score!

How about the agreement between the LLM specificity scores that already came with the dataset (i.e., column `score_specificity_llm`) and the human expert-coded scores?

Note that `score_specificity_llm` is based on prompts that were carefully engineered by Gabrielle.

In [54]:
best_llm_specificity_scores = df.best_llm_specificity_score[:10].tolist()
print("Krippendorff's Alpha:", compute_krippendorff_alpha(best_llm_specificity_scores, expert_specificity_scores))

Krippendorff's Alpha: 0.8874074074074074


Wow! Much better!

## Exercise: Try a different dataset of choice!